- PREPROCESSING NLP DES TICKETS
- Projet NLP - Classification de Tickets de Support

In [32]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

In [33]:
print("\n1. CHARGEMENT DES DONNÉES")
print("-"*80)

# Modifier ce chemin selon votre structure
df = pd.read_csv('../data/raw/dataset.csv')

print(f"✓ Dataset chargé : {len(df):,} tickets")

# Créer un ID si nécessaire
if 'id' not in df.columns:
    df['id'] = range(len(df))
    print("✓ Colonne 'id' créée")


1. CHARGEMENT DES DONNÉES
--------------------------------------------------------------------------------
✓ Dataset chargé : 20,000 tickets
✓ Colonne 'id' créée


In [34]:
print("\n2. FUSION DES CHAMPS TEXTUELS")
print("-"*80)

# Remplir les valeurs manquantes
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')

# Créer la colonne combinée
df['text_combined'] = df['subject'] + ' ' + df['body']

print(f"✓ Colonnes 'subject' et 'body' fusionnées dans 'text_combined'")
print(f"\nExemple :")
print(df['text_combined'].iloc[0][:150] + "...")


2. FUSION DES CHAMPS TEXTUELS
--------------------------------------------------------------------------------
✓ Colonnes 'subject' et 'body' fusionnées dans 'text_combined'

Exemple :
Unvorhergesehener Absturz der Datenanalyse-Plattform Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe...


In [35]:
print("\n3. CONFIGURATION DES STOPWORDS")
print("-"*80)

# Stopwords anglais (les plus courants)
stopwords_en = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours',
    'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers',
    'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves',
    'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does',
    'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
    'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into',
    'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down',
    'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once'
}

# Stopwords allemands (les plus courants)
stopwords_de = {
    'der', 'die', 'das', 'den', 'dem', 'des', 'ein', 'eine', 'einer', 'eines', 'einem',
    'einen', 'ich', 'du', 'er', 'sie', 'es', 'wir', 'ihr', 'und', 'oder', 'aber', 'nicht',
    'mit', 'von', 'zu', 'im', 'am', 'um', 'ist', 'sind', 'war', 'waren', 'hat', 'haben',
    'wird', 'werden', 'kann', 'können', 'muss', 'müssen', 'auf', 'für', 'als', 'bei',
    'nach', 'vor', 'über', 'unter', 'durch', 'an', 'aus', 'in', 'zwischen', 'auch',
    'wenn', 'dann', 'noch', 'mehr', 'sehr', 'so', 'wie', 'dass', 'daß', 'sich', 'sein'
}

# Combiner les stopwords
stopwords = stopwords_en.union(stopwords_de)
print(f"✓ Stopwords définis : {len(stopwords_en)} (EN) + {len(stopwords_de)} (DE) = {len(stopwords)} total")



3. CONFIGURATION DES STOPWORDS
--------------------------------------------------------------------------------
✓ Stopwords définis : 93 (EN) + 66 (DE) = 156 total


In [36]:
print("\n4. DÉFINITION DE LA FONCTION DE NETTOYAGE")
print("-"*80)

def clean_text(text, remove_stopwords=True):
    """
    Pipeline complet de nettoyage du texte.
    
    Étapes :
    1. Conversion en minuscules
    2. Suppression URLs, emails, numéros de téléphone
    3. Suppression des balises HTML
    4. Remplacement des nombres par token NUM
    5. Suppression de la ponctuation
    6. Suppression des caractères spéciaux
    7. Tokenisation simple
    8. Suppression des stopwords (optionnel)
    9. Normalisation des espaces
    
    Args:
        text (str): Texte à nettoyer
        remove_stopwords (bool): Supprimer les stopwords
    
    Returns:
        str: Texte nettoyé
    """
    if not text or pd.isna(text):
        return ""
    
    # 1. Conversion en minuscules
    text = text.lower()
    
    # 2. Suppression des URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Suppression des emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # 4. Suppression des numéros de téléphone et balises
    text = re.sub(r'<[^>]+>', '', text)
    
    # 5. Suppression des balises HTML
    text = re.sub(r'<.*?>', '', text)
    
    # 6. Remplacement des nombres par un token
    text = re.sub(r'\d+', ' NUM ', text)
    
    # 7. Suppression de la ponctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 8. Suppression des caractères spéciaux (garder lettres + espaces)
    text = re.sub(r'[^a-zäöüßàéèêëîïôùûüç\s]', '', text)
    
    # 9. Tokenisation simple (split par espaces)
    tokens = text.split()
    
    # 10. Suppression des stopwords
    if remove_stopwords:
        tokens = [word for word in tokens if word not in stopwords and len(word) > 1]
    else:
        tokens = [word for word in tokens if len(word) > 1]
    
    # 11. Rejoindre les tokens
    cleaned_text = ' '.join(tokens)
    
    # 12. Normalisation des espaces multiples
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

print("✓ Fonction clean_text() définie")
print("  Pipeline : lowercase → URLs → emails → HTML → ponctuation → stopwords → normalisation")



4. DÉFINITION DE LA FONCTION DE NETTOYAGE
--------------------------------------------------------------------------------
✓ Fonction clean_text() définie
  Pipeline : lowercase → URLs → emails → HTML → ponctuation → stopwords → normalisation


In [37]:
print("\n5. TEST DE LA FONCTION DE NETTOYAGE")
print("-"*80)

print("\nEXEMPLES AVANT/APRÈS :")
for i in range(min(3, len(df))):
    original = df['text_combined'].iloc[i]
    cleaned = clean_text(original, remove_stopwords=True)
    
    print(f"\n[Exemple {i+1}]")
    print(f"Type: {df['type'].iloc[i]} | Langue: {df['language'].iloc[i]}")
    print(f"\nAVANT ({len(original)} chars):")
    print(original[:150] + "...")
    print(f"\nAPRÈS ({len(cleaned)} chars):")
    print(cleaned[:150] + "...")
    print("-"*80)


5. TEST DE LA FONCTION DE NETTOYAGE
--------------------------------------------------------------------------------

EXEMPLES AVANT/APRÈS :

[Exemple 1]
Type: Incident | Langue: de

AVANT (305 chars):
Unvorhergesehener Absturz der Datenanalyse-Plattform Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe...

APRÈS (236 chars):
unvorhergesehener absturz datenanalyseplattform datenanalyseplattform brach unerwartet ab da speicheroberfläche gering habe versucht laravel meinen ma...
--------------------------------------------------------------------------------

[Exemple 2]
Type: Request | Langue: en

AVANT (250 chars):
Customer Support Inquiry Seeking information on digital strategies that can aid in brand growth and details on the available services. Looking forward...

APRÈS (192 chars):
customer support inquiry seeking information digital strategies can aid brand growth details available services looking forward learning more help bus...


In [38]:
print("\n6. APPLICATION DU PREPROCESSING")
print("-"*80)

print("Traitement en cours...")
print(f"  Nombre de tickets à traiter : {len(df):,}")
print("  (Cela peut prendre 1-2 minutes...)")

import time
start_time = time.time()

# Appliquer le nettoyage
df['text_cleaned'] = df['text_combined'].apply(lambda x: clean_text(x, remove_stopwords=True))

elapsed_time = time.time() - start_time
print(f"\n✓ Preprocessing terminé en {elapsed_time:.1f} secondes")



6. APPLICATION DU PREPROCESSING
--------------------------------------------------------------------------------
Traitement en cours...
  Nombre de tickets à traiter : 20,000
  (Cela peut prendre 1-2 minutes...)

✓ Preprocessing terminé en 2.3 secondes


In [39]:
print("\n7. VÉRIFICATION DE LA QUALITÉ DU PREPROCESSING")
print("-"*80)

# Calculer le nombre de mots
df['word_count_cleaned'] = df['text_cleaned'].str.split().str.len()

# Identifier les textes vides
empty_texts = df[df['text_cleaned'].str.strip() == '']
print(f"\n⚠ Textes vides après nettoyage : {len(empty_texts)} ({len(empty_texts)/len(df)*100:.2f}%)")

# Marquer les textes vides
df['is_empty'] = df['text_cleaned'].str.strip() == ''

# Textes très courts
short_texts = df[df['word_count_cleaned'] < 3]
print(f"⚠ Textes avec < 3 mots : {len(short_texts)} ({len(short_texts)/len(df)*100:.2f}%)")



7. VÉRIFICATION DE LA QUALITÉ DU PREPROCESSING
--------------------------------------------------------------------------------

⚠ Textes vides après nettoyage : 0 (0.00%)
⚠ Textes avec < 3 mots : 5 (0.03%)


In [40]:
print("\n8. STATISTIQUES APRÈS PREPROCESSING")
print("-"*80)

print(f"\nSTATISTIQUES :")
print(f"  Nombre moyen de mots : {df['word_count_cleaned'].mean():.1f}")
print(f"  Médiane : {df['word_count_cleaned'].median():.0f}")
print(f"  Min : {df['word_count_cleaned'].min():.0f}")
print(f"  Max : {df['word_count_cleaned'].max():.0f}")
print(f"  Écart-type : {df['word_count_cleaned'].std():.1f}")

# Distribution par type
print(f"\nDISTRIBUTION PAR TYPE :")
for ticket_type in df['type'].unique():
    subset = df[df['type'] == ticket_type]
    avg_words = subset['word_count_cleaned'].mean()
    count = len(subset)
    print(f"  {ticket_type:12s} : {count:5,} tickets (moy: {avg_words:.1f} mots)")



8. STATISTIQUES APRÈS PREPROCESSING
--------------------------------------------------------------------------------

STATISTIQUES :
  Nombre moyen de mots : 38.9
  Médiane : 36
  Min : 1
  Max : 183
  Écart-type : 21.7

DISTRIBUTION PAR TYPE :
  Incident     : 7,978 tickets (moy: 37.4 mots)
  Request      : 5,763 tickets (moy: 40.7 mots)
  Problem      : 4,184 tickets (moy: 37.2 mots)
  Change       : 2,075 tickets (moy: 43.0 mots)


In [41]:
print("\n9. VISUALISATION : AVANT vs APRÈS")
print("-"*80)

# Calculer les stats avant nettoyage
df['word_count_original'] = df['text_combined'].str.split().str.len()

# Créer le graphique de comparaison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribution AVANT
axes[0].hist(df['word_count_original'], bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0].axvline(df['word_count_original'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Moyenne: {df["word_count_original"].mean():.1f}')
axes[0].set_title('AVANT Preprocessing', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Nombre de mots', fontsize=11)
axes[0].set_ylabel('Fréquence', fontsize=11)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribution APRÈS
axes[1].hist(df['word_count_cleaned'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1].axvline(df['word_count_cleaned'].mean(), color='green', linestyle='--', linewidth=2, 
                label=f'Moyenne: {df["word_count_cleaned"].mean():.1f}')
axes[1].set_title('APRÈS Preprocessing', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Nombre de mots', fontsize=11)
axes[1].set_ylabel('Fréquence', fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()

# Créer le dossier si nécessaire
os.makedirs('reports/eda', exist_ok=True)
plt.savefig('reports/eda/preprocessing_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Graphique sauvegardé : reports/eda/preprocessing_comparison.png")
plt.close()

# Calculer la réduction
reduction = ((df['word_count_original'].mean() - df['word_count_cleaned'].mean()) / df['word_count_original'].mean()) * 100
print(f"✓ Réduction moyenne : {reduction:.1f}%")



9. VISUALISATION : AVANT vs APRÈS
--------------------------------------------------------------------------------
✓ Graphique sauvegardé : reports/eda/preprocessing_comparison.png
✓ Réduction moyenne : 37.5%


In [42]:
print("\n10. EXEMPLES DE TRANSFORMATION COMPLÈTE")
print("-"*80)

for i in range(min(5, len(df))):
    print(f"\n[Exemple {i+1}]")
    print(f"Type: {df.iloc[i]['type']} | Langue: {df.iloc[i]['language']} | Priorité: {df.iloc[i]['priority']}")
    print(f"\nAVANT ({df.iloc[i]['word_count_original']} mots):")
    print(df.iloc[i]['text_combined'][:150] + "...")
    print(f"\nAPRÈS ({df.iloc[i]['word_count_cleaned']} mots):")
    print(df.iloc[i]['text_cleaned'][:150] + "...")
    print("-"*80)


10. EXEMPLES DE TRANSFORMATION COMPLÈTE
--------------------------------------------------------------------------------

[Exemple 1]
Type: Incident | Langue: de | Priorité: low

AVANT (42 mots):
Unvorhergesehener Absturz der Datenanalyse-Plattform Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu gering war. Ich habe...

APRÈS (26 mots):
unvorhergesehener absturz datenanalyseplattform datenanalyseplattform brach unerwartet ab da speicheroberfläche gering habe versucht laravel meinen ma...
--------------------------------------------------------------------------------

[Exemple 2]
Type: Request | Langue: en | Priorité: medium

AVANT (41 mots):
Customer Support Inquiry Seeking information on digital strategies that can aid in brand growth and details on the available services. Looking forward...

APRÈS (26 mots):
customer support inquiry seeking information digital strategies can aid brand growth details available services looking forward learning more help 

In [43]:
print("\n11. SAUVEGARDE DES DONNÉES PREPROCESSED")
print("-"*80)

# Sélectionner les colonnes importantes
output_cols = ['id', 'text_combined', 'text_cleaned', 'type', 'language', 
               'priority', 'queue', 'word_count_cleaned', 'is_empty']
output_df = df[output_cols]

# Créer le répertoire si nécessaire
output_dir = 'data/processed'
os.makedirs(output_dir, exist_ok=True)

# Sauvegarder
output_path = f'{output_dir}/preprocessed_data.csv'
output_df.to_csv(output_path, index=False)

print(f"✓ Données sauvegardées : {output_path}")
print(f"  Nombre de lignes : {len(output_df):,}")
print(f"  Colonnes : {list(output_df.columns)}")

print("\nÉCHANTILLON DES DONNÉES SAUVEGARDÉES :")
print(output_df.head())



11. SAUVEGARDE DES DONNÉES PREPROCESSED
--------------------------------------------------------------------------------
✓ Données sauvegardées : data/processed/preprocessed_data.csv
  Nombre de lignes : 20,000
  Colonnes : ['id', 'text_combined', 'text_cleaned', 'type', 'language', 'priority', 'queue', 'word_count_cleaned', 'is_empty']

ÉCHANTILLON DES DONNÉES SAUVEGARDÉES :
   id                                      text_combined  \
0   0  Unvorhergesehener Absturz der Datenanalyse-Pla...   
1   1  Customer Support Inquiry Seeking information o...   
2   2  Data Analytics for Investment I am contacting ...   
3   3  Krankenhaus-Dienstleistung-Problem Ein Medien-...   
4   4  Security Dear Customer Support, I am reaching ...   

                                        text_cleaned      type language  \
0  unvorhergesehener absturz datenanalyseplattfor...  Incident       de   
1  customer support inquiry seeking information d...   Request       en   
2  data analytics investment conta

In [44]:
print("\n" + "="*80)
print("RÉSUMÉ FINAL")
print("="*80)

print(f"\n✓ TRAITEMENT :")
print(f"  • Tickets traités : {len(df):,}")
print(f"  • Textes vides après nettoyage : {len(empty_texts)} ({len(empty_texts)/len(df)*100:.2f}%)")
print(f"  • Textes courts (<3 mots) : {len(short_texts)} ({len(short_texts)/len(df)*100:.2f}%)")

print(f"\n✓ STATISTIQUES :")
print(f"  • Nombre moyen de mots (avant) : {df['word_count_original'].mean():.1f}")
print(f"  • Nombre moyen de mots (après) : {df['word_count_cleaned'].mean():.1f}")
print(f"  • Réduction : {reduction:.1f}%")

print(f"\n✓ PIPELINE APPLIQUÉ :")
print(f"  1. Fusion subject + body")
print(f"  2. Conversion en minuscules")
print(f"  3. Suppression URLs, emails, numéros")
print(f"  4. Suppression ponctuation")
print(f"  5. Suppression stopwords (EN + DE)")
print(f"  6. Normalisation des espaces")

print(f"\nFICHIERS DE SORTIE :")
print(f"  • {output_path}")
print(f"  • reports/eda/preprocessing_comparison.png")

print(f"\nPROCHAINES ÉTAPES :")
print(f"  1. ✓ EDA terminée")
print(f"  2. ✓ Preprocessing terminé")
print(f"  3. → Génération des embeddings (Phase 2)")
print(f"  4. → Entraînement du modèle (Phase 3)")

print("\n" + "="*80)
print("PREPROCESSING TERMINÉ AVEC SUCCÈS ✓")
print("="*80)


RÉSUMÉ FINAL

✓ TRAITEMENT :
  • Tickets traités : 20,000
  • Textes vides après nettoyage : 0 (0.00%)
  • Textes courts (<3 mots) : 5 (0.03%)

✓ STATISTIQUES :
  • Nombre moyen de mots (avant) : 62.2
  • Nombre moyen de mots (après) : 38.9
  • Réduction : 37.5%

✓ PIPELINE APPLIQUÉ :
  1. Fusion subject + body
  2. Conversion en minuscules
  3. Suppression URLs, emails, numéros
  4. Suppression ponctuation
  5. Suppression stopwords (EN + DE)
  6. Normalisation des espaces

FICHIERS DE SORTIE :
  • data/processed/preprocessed_data.csv
  • reports/eda/preprocessing_comparison.png

PROCHAINES ÉTAPES :
  1. ✓ EDA terminée
  2. ✓ Preprocessing terminé
  3. → Génération des embeddings (Phase 2)
  4. → Entraînement du modèle (Phase 3)

PREPROCESSING TERMINÉ AVEC SUCCÈS ✓
